# Búsquedas avanzadas

Variantes de la búsqueda binaria y de la búsqueda en grafos para casos concretos:

- **Ternaria**: el máximo de una función unimodal descartando un tercio en cada paso, $O(\log n)$ evaluaciones.
- **Exponencial (galopante)**: dobla el salto hasta pasarse y luego bisecciona; $O(\log i)$ si el objetivo está en la posición $i$, sin conocer la longitud.
- **Interpolación**: estima la posición por proporción; $O(\log \log n)$ de media con datos uniformes, pero $O(n)$ en el peor caso.
- **Por saltos**: avanza de $\sqrt{n}$ en $\sqrt{n}$ y termina en lineal, $O(\sqrt{n})$.
- **Bidireccional**: dos BFS que se encuentran a mitad de camino; con ramificación $b$ y distancia $d$ expande $O(b^{d/2})$ nodos en lugar de $O(b^d)$.
- **IDA\***: profundización iterativa con la cota $f = g + h$ de A\*; misma solución óptima con memoria $O(d)$ en lugar de $O(b^d)$, a cambio de reexpandir nodos.
- **Binaria sobre la respuesta**: si un predicado es monótono en la respuesta, se busca la frontera en $O(\log R)$ comprobaciones.

Todas se comprueban contra la búsqueda lineal, BFS o la fuerza bruta.

## 1. Búsqueda ternaria en enteros

Una guagua turística cobra $p$ euros y vende $500 - 2p$ billetes. Encontrar con búsqueda ternaria el precio entero en $[0, 250]$ que maximiza los ingresos, contar las evaluaciones y comprobarlo contra recorrer todos los precios. Ver también que con una meseta (función no estrictamente unimodal) la ternaria puede fallar.

In [1]:
def ternaria_entera(f, izq, der):
    evaluaciones = 0
    while der - izq > 2:
        m1, m2 = izq + (der - izq) // 3, der - (der - izq) // 3
        evaluaciones += 2
        if f(m1) < f(m2):
            izq = m1 + 1
        else:
            der = m2
    candidatos = range(izq, der + 1)
    return max(candidatos, key=f), evaluaciones + len(candidatos)


def ingresos(p):
    return p * (500 - 2 * p)


mejor, evals = ternaria_entera(ingresos, 0, 250)
print(f"precio óptimo {mejor} €, ingresos {ingresos(mejor)} €, {evals} evaluaciones frente a 251")
assert mejor == max(range(251), key=ingresos)

import random

random.seed(1)
for _ in range(500):
    a, b = sorted(random.sample(range(-1000, 1000), 2))
    c = random.randint(a, b)
    f = lambda x, c=c: -abs(x - c) * 3 + random.Random(x).random()  # unimodal con ruido menor que la pendiente
    assert ternaria_entera(f, a, b)[0] == max(range(a, b + 1), key=f)


def meseta(x):
    return 10 if x == 70 else 0


print("con meseta:", ternaria_entera(meseta, 0, 100)[0], "| real:", max(range(101), key=meseta))

precio óptimo 125 €, ingresos 31250 €, 26 evaluaciones frente a 251


con meseta: 0 | real: 70


## 2. Búsqueda ternaria en reales y sección áurea

De una chapa de $a \times b$ cm se cortan cuadrados de lado $x$ en las esquinas para doblar una caja sin tapa de volumen $V(x) = x(a-2x)(b-2x)$. Hallar el $x$ óptimo con búsqueda ternaria en reales y compararlo con la solución exacta de $V'(x) = 0$. Repetirlo con la **sección áurea**, que reutiliza uno de los dos puntos en cada paso, y comparar cuántas evaluaciones necesita cada una para la misma precisión.

In [2]:
import math


def ternaria_real(f, izq, der, tolerancia=1e-9):
    evaluaciones = 0
    while der - izq > tolerancia:
        m1, m2 = izq + (der - izq) / 3, der - (der - izq) / 3
        evaluaciones += 2
        if f(m1) < f(m2):
            izq = m1
        else:
            der = m2
    return (izq + der) / 2, evaluaciones


def seccion_aurea(f, izq, der, tolerancia=1e-9):
    r = (math.sqrt(5) - 1) / 2
    m1, m2 = der - r * (der - izq), izq + r * (der - izq)
    f1, f2 = f(m1), f(m2)
    evaluaciones = 2
    while der - izq > tolerancia:
        if f1 < f2:
            izq, m1, f1 = m1, m2, f2
            m2 = izq + r * (der - izq)
            f2 = f(m2)
        else:
            der, m2, f2 = m2, m1, f1
            m1 = der - r * (der - izq)
            f1 = f(m1)
        evaluaciones += 1
    return (izq + der) / 2, evaluaciones


a, b = 60, 40
volumen = lambda x: x * (a - 2 * x) * (b - 2 * x)
exacto = ((a + b) - math.sqrt(a * a - a * b + b * b)) / 6
print(f"{'método':<15}{'x (cm)':>14}{'V (cm³)':>14}{'evaluaciones':>14}")
for nombre, metodo in [("ternaria", ternaria_real), ("sección áurea", seccion_aurea)]:
    x, evals = metodo(volumen, 0, min(a, b) / 2)
    print(f"{nombre:<15}{x:>14.8f}{volumen(x):>14.3f}{evals:>14}")
    assert abs(x - exacto) < 1e-6
print(f"{'exacto':<15}{exacto:>14.8f}{volumen(exacto):>14.3f}")

método                 x (cm)       V (cm³)  evaluaciones
ternaria           7.84749559      8450.447           118
sección áurea      7.84749550      8450.447            52
exacto             7.84749563      8450.447


## 3. Búsqueda exponencial sin conocer la longitud

Un registro de llegadas a un puerto está ordenado por hora, pero solo se puede consultar por posición y lanza `IndexError` al pasarse del final; no tiene `len`. Escribir una búsqueda galopante (saltos 1, 2, 4, 8… y bisección en el último intervalo) que devuelva la primera posición con un valor mayor o igual que el buscado, contar los accesos y comprobar que crecen con $\log i$, donde $i$ es la posición del resultado, y no con el tamaño total.

In [3]:
class Registro:
    def __init__(self, datos):
        self._datos = datos
        self.accesos = 0

    def __getitem__(self, i):
        self.accesos += 1
        return self._datos[i]  # IndexError al pasarse


def leer(registro, i):
    try:
        return registro[i]
    except IndexError:
        return math.inf


def galopante(registro, x):
    if leer(registro, 0) >= x:
        return 0
    fin = 1
    while leer(registro, fin) < x:
        fin *= 2
    izq, der = fin // 2 + 1, fin  # registro[fin // 2] < x <= registro[fin]
    while izq < der:
        medio = (izq + der) // 2
        if leer(registro, medio) < x:
            izq = medio + 1
        else:
            der = medio
    return izq


import bisect

random.seed(3)
llegadas = sorted(random.sample(range(10**7), 1_000_000))
print(f"{'posición':>10}{'accesos':>10}{'2·log2(i+1)':>13}")
for objetivo in (llegadas[5], llegadas[100], llegadas[10_000], llegadas[999_999], 10**7):
    r = Registro(llegadas)
    pos = galopante(r, objetivo)
    print(f"{pos:>10}{r.accesos:>10}{2 * math.log2(pos + 1):>13.1f}")
    assert pos == bisect.bisect_left(llegadas, objetivo)

for _ in range(2000):
    datos = sorted(random.choices(range(100), k=random.randint(0, 40)))
    x = random.randint(-5, 105)
    assert galopante(Registro(datos), x) == next((i for i, v in enumerate(datos) if v >= x), len(datos))

  posición   accesos  2·log2(i+1)
         5         7          5.2
       100        15         13.3
     10000        29         26.6
    999999        41         39.9
   1000000        41         39.9


## 4. Intersección galopante

Intersecar una lista ordenada de $m$ códigos con otra de $n = 100\,000$ de tres formas: mezcla lineal de las dos ($m + n$ comparaciones), `bisect` en toda la lista larga por cada elemento ($m \log_2 n$) y galope desde la última posición encontrada ($\approx 2m \log_2(n/m)$). Contar comparaciones para varios $m$ y comprobar que el galope nunca queda lejos de la mejor de las otras dos: con $m$ pequeño se comporta como `bisect` y con $m$ cercano a $n$, como la mezcla.

In [4]:
class Comparable:
    cuenta = 0
    __slots__ = ("v",)

    def __init__(self, v):
        self.v = v

    def __lt__(self, otro):
        Comparable.cuenta += 1
        return self.v < otro.v

    def __eq__(self, otro):
        return self.v == otro.v


def interseccion_mezcla(a, b):
    i = j = 0
    salida = []
    while i < len(a) and j < len(b):
        if a[i] < b[j]:
            i += 1
        elif b[j] < a[i]:
            j += 1
        else:
            salida.append(a[i])
            i, j = i + 1, j + 1
    return salida


def interseccion_bisect(a, b):
    salida = []
    for x in a:
        j = bisect.bisect_left(b, x)
        if j < len(b) and not x < b[j]:
            salida.append(x)
    return salida


def interseccion_galope(a, b):
    salida, j = [], 0
    for x in a:
        salto = 1
        while j + salto < len(b) and b[j + salto] < x:
            salto *= 2
        j = bisect.bisect_left(b, x, j + salto // 2, min(j + salto + 1, len(b)))
        if j < len(b) and not x < b[j]:
            salida.append(x)
    return salida


random.seed(4)
largo = sorted(random.sample(range(1_000_000), 100_000))
b = [Comparable(v) for v in largo]
metodos = {"mezcla": interseccion_mezcla, "bisect": interseccion_bisect, "galope": interseccion_galope}
print(f"{'m':>7}" + "".join(f"{nombre:>10}" for nombre in metodos) + f"{'2m·log2(n/m)':>14}")
for m in (10, 300, 3_000, 30_000):
    corto = sorted(set(random.sample(largo, m // 2) + random.sample(range(1_000_000), m - m // 2)))
    a = [Comparable(v) for v in corto]
    esperado = sorted(set(corto) & set(largo))
    cuentas = {}
    for nombre, metodo in metodos.items():
        Comparable.cuenta = 0
        assert [c.v for c in metodo(a, b)] == esperado
        cuentas[nombre] = Comparable.cuenta
    print(f"{m:>7}" + "".join(f"{c:>10}" for c in cuentas.values()) + f"{2 * m * math.log2(len(largo) / m):>14.0f}")
    assert cuentas["galope"] <= 2.5 * min(cuentas["mezcla"], cuentas["bisect"])

      m    mezcla    bisect    galope  2m·log2(n/m)
     10    158243       175       266           266


    300    200128      5307      5131          5028
   3000    201282     52992     32470         30353


  30000    213529    526364    170620        104218


## 5. Búsqueda por interpolación

En lugar de mirar el centro, estimar la posición con $pos = izq + \frac{(x - a_{izq})(der - izq)}{a_{der} - a_{izq}}$. Contar las sondas de media para datos uniformes de distintos tamaños y compararlas con la búsqueda binaria y con $\log_2 \log_2 n$.

In [5]:
def interpolacion(a, x):
    izq, der, sondas = 0, len(a) - 1, 0
    while izq <= der and a[izq] <= x <= a[der]:
        sondas += 1
        if a[der] == a[izq]:
            pos = izq
        else:
            pos = izq + (x - a[izq]) * (der - izq) // (a[der] - a[izq])
        if a[pos] == x:
            return pos, sondas
        if a[pos] < x:
            izq = pos + 1
        else:
            der = pos - 1
    return -1, sondas


def binaria(a, x):
    izq, der, sondas = 0, len(a) - 1, 0
    while izq <= der:
        sondas += 1
        medio = (izq + der) // 2
        if a[medio] == x:
            return medio, sondas
        if a[medio] < x:
            izq = medio + 1
        else:
            der = medio - 1
    return -1, sondas


rng = random.Random(5)
print(f"{'n':>9}{'interpolación':>15}{'binaria':>9}{'log2 log2 n':>13}{'log2 n':>8}")
for n in (10**3, 10**4, 10**5, 10**6):
    datos = sorted(rng.sample(range(10 * n), n))
    objetivos = rng.sample(datos, 300) + [rng.randrange(10 * n) for _ in range(100)]
    presentes = set(datos)
    s_int = s_bin = 0
    for x in objetivos:
        (p1, s1), (p2, s2) = interpolacion(datos, x), binaria(datos, x)
        esperado = bisect.bisect_left(datos, x) if x in presentes else -1
        assert p1 == p2 == esperado
        s_int, s_bin = s_int + s1, s_bin + s2
    print(f"{n:>9}{s_int / len(objetivos):>15.2f}{s_bin / len(objetivos):>9.2f}{math.log2(math.log2(n)):>13.2f}{math.log2(n):>8.2f}")

        n  interpolación  binaria  log2 log2 n  log2 n
     1000           3.58     9.11         3.32    9.97
    10000           3.54    12.56         3.73   13.29
   100000           3.79    15.86         4.05   16.61


  1000000           3.93    19.18         4.32   19.93


## 6. El peor caso de la interpolación

Con datos muy sesgados la estimación se equivoca sistemáticamente. Construir listas de la forma $0, 1, 2, \dots, n-2, 10^{12}$ y buscar el penúltimo valor: cada sonda avanza una sola posición. Contar las sondas frente a la binaria para varios $n$ y comprobar que crecen linealmente. Probar también con valores que crecen exponencialmente ($2^i$).

In [6]:
print(f"{'datos':<12}{'n':>7}{'interpolación':>15}{'binaria':>9}")
for n in (100, 1000, 10_000):
    sesgados = list(range(n - 1)) + [10**12]
    exponenciales = [2**i for i in range(n)]
    for nombre, datos, x in [("con atípico", sesgados, n - 2), ("2^i", exponenciales, 2 ** (n // 3))]:
        (p1, s1), (p2, s2) = interpolacion(datos, x), binaria(datos, x)
        print(f"{nombre:<12}{n:>7}{s1:>15}{s2:>9}")
        assert p1 == p2 == datos.index(x)
assert interpolacion(list(range(9_999)) + [10**12], 9_998)[1] == 9_999

datos             n  interpolación  binaria
con atípico     100             99        6
2^i             100             34        5
con atípico    1000            999        9
2^i            1000            334       10
con atípico   10000           9999       13
2^i           10000           3334       11


## 7. Búsqueda por saltos

Recorrer una lista ordenada a saltos de $m$ posiciones hasta encontrar un bloque cuyo último elemento sea mayor o igual que el buscado, y terminar con una búsqueda lineal dentro del bloque. Medir el peor número de comparaciones sobre todos los objetivos posibles para varios $m$ y comprobar que el mínimo está en $m \approx \sqrt{n}$, con un coste cercano a $2\sqrt{n}$.

In [7]:
def por_saltos(a, x, m):
    comparaciones, n = 0, len(a)
    inicio = 0
    while inicio < n:
        fin = min(inicio + m, n) - 1
        comparaciones += 1
        if a[fin] >= x:
            break
        inicio += m
    else:
        return -1, comparaciones
    for i in range(inicio, fin + 1):
        comparaciones += 1
        if a[i] == x:
            return i, comparaciones
        if a[i] > x:
            break
    return -1, comparaciones


n = 2500
datos = list(range(0, 2 * n, 2))
print(f"{'m':>5}{'peor caso':>11}{'n/m + m':>9}")
peores = {}
for m in (5, 25, 40, 50, 60, 100, 500):
    peor = 0
    for x in range(-1, 2 * n + 1):
        pos, comparaciones = por_saltos(datos, x, m)
        assert pos == (x // 2 if x % 2 == 0 and 0 <= x < 2 * n else -1)
        peor = max(peor, comparaciones)
    peores[m] = peor
    print(f"{m:>5}{peor:>11}{n // m + m:>9}")
print(f"√n = {math.isqrt(n)}, 2√n = {2 * math.isqrt(n)}")
assert min(peores, key=peores.get) == math.isqrt(n)

    m  peor caso  n/m + m
    5        505      505
   25        125      125
   40        102      102
   50        100      100


   60        101      101
  100        125      125


  500        505      505
√n = 50, 2√n = 100


## 8. Búsqueda bidireccional frente a BFS

Generar un grafo aleatorio no dirigido de 50 000 nodos (cada nodo se une a 3 vecinos al azar) y una rejilla de $300 \times 300$. Buscar la distancia entre parejas de nodos con BFS y con búsqueda bidireccional (expandiendo cada vez el nivel completo de la frontera más pequeña), contar los nodos expandidos por cada una y comprobar que las distancias coinciden.

In [8]:
from collections import deque


def bfs(grafo, origen, destino):
    distancia, cola, expandidos = {origen: 0}, deque([origen]), 0
    while cola:
        u = cola.popleft()
        expandidos += 1
        if u == destino:
            return distancia[u], expandidos
        for v in grafo[u]:
            if v not in distancia:
                distancia[v] = distancia[u] + 1
                cola.append(v)
    return None, expandidos


def bidireccional(grafo, origen, destino):
    if origen == destino:
        return 0, 0
    dist = ({origen: 0}, {destino: 0})
    fronteras = ([origen], [destino])
    expandidos = 0
    while fronteras[0] and fronteras[1]:
        lado = 0 if len(fronteras[0]) <= len(fronteras[1]) else 1
        propio, otro = dist[lado], dist[1 - lado]
        mejor, siguiente = math.inf, []
        for u in fronteras[lado]:  # el nivel completo, para que el primer encuentro sea óptimo
            expandidos += 1
            for v in grafo[u]:
                if v in otro:
                    mejor = min(mejor, propio[u] + 1 + otro[v])
                if v not in propio:
                    propio[v] = propio[u] + 1
                    siguiente.append(v)
        if mejor < math.inf:
            return mejor, expandidos
        fronteras = (siguiente, fronteras[1]) if lado == 0 else (fronteras[0], siguiente)
    return None, expandidos


def grafo_aleatorio(n, enlaces, semilla):
    rng = random.Random(semilla)
    grafo = [[] for _ in range(n)]
    for u in range(n):
        for v in rng.sample(range(n), enlaces):
            if v != u:
                grafo[u].append(v)
                grafo[v].append(u)
    return grafo


def rejilla(lado):
    grafo = [[] for _ in range(lado * lado)]
    for f in range(lado):
        for c in range(lado):
            for df, dc in ((0, 1), (1, 0)):
                if f + df < lado and c + dc < lado:
                    u, v = f * lado + c, (f + df) * lado + c + dc
                    grafo[u].append(v)
                    grafo[v].append(u)
    return grafo


rng = random.Random(2024)
print(f"{'grafo':<10}{'distancia':>10}{'BFS':>9}{'bidireccional':>15}{'ahorro':>8}")
for nombre, grafo in [("aleatorio", grafo_aleatorio(50_000, 3, 8)), ("rejilla", rejilla(300))]:
    for _ in range(4):
        s, t = rng.sample(range(len(grafo)), 2)
        (d1, e1), (d2, e2) = bfs(grafo, s, t), bidireccional(grafo, s, t)
        print(f"{nombre:<10}{d1:>10}{e1:>9}{e2:>15}{e1 / e2:>7.1f}×")
        assert d1 == d2

pequeno = grafo_aleatorio(300, 1, 9)
for s in range(0, 300, 7):
    for t in range(0, 300, 11):
        assert bfs(pequeno, s, t)[0] == bidireccional(pequeno, s, t)[0]

grafo      distancia      BFS  bidireccional  ahorro


aleatorio          8    47765            417  114.5×
aleatorio          6    14561             79  184.3×


aleatorio          7    36118            242  149.2×


aleatorio          7    42521            181  234.9×
rejilla          139    27714          14501    1.9×
rejilla          417    89158          76106    1.2×
rejilla           72    10244           5042    2.0×
rejilla          140    36359          19322    1.9×


## 9. A\* sobre el 8-puzle

Calcular con un BFS desde la meta la distancia exacta de los 181 440 tableros resolubles del 8-puzle. Escribir después A\* con la distancia Manhattan como heurística, resolver tableros obtenidos con paseos aleatorios desde la meta y comprobar que la longitud de la solución es la óptima; anotar los nodos expandidos y el máximo de nodos guardados en memoria (abiertos más cerrados). En *Espacio de estados* (ejercicio 8) se comparan heurísticas con primero el mejor, que no garantiza el óptimo.

In [9]:
import heapq

META = (1, 2, 3, 4, 5, 6, 7, 8, 0)
VECINOS = [[j for j in (i - 3, i + 3, i - 1 if i % 3 else -1, i + 1 if i % 3 < 2 else -1) if 0 <= j < 9] for i in range(9)]


def sucesores(tablero):
    hueco = tablero.index(0)
    for j in VECINOS[hueco]:
        nuevo = list(tablero)
        nuevo[hueco], nuevo[j] = nuevo[j], 0
        yield tuple(nuevo)


def distancias_desde_meta():
    distancia, cola = {META: 0}, deque([META])
    while cola:
        u = cola.popleft()
        for v in sucesores(u):
            if v not in distancia:
                distancia[v] = distancia[u] + 1
                cola.append(v)
    return distancia


EXACTA = distancias_desde_meta()
print(len(EXACTA), "tableros alcanzables; distancia máxima:", max(EXACTA.values()))
assert len(EXACTA) == 9 * 8 * 7 * 6 * 5 * 4 * 3 * 2 // 2

DESTINO = {v: divmod(i, 3) for i, v in enumerate(META)}


def manhattan(tablero):
    total = 0
    for i, v in enumerate(tablero):
        if v:
            f, c = divmod(i, 3)
            mf, mc = DESTINO[v]
            total += abs(f - mf) + abs(c - mc)
    return total


def a_estrella(inicio):
    g = {inicio: 0}
    abiertos = [(manhattan(inicio), 0, inicio)]
    cerrados, expandidos = set(), 0
    while abiertos:
        _, coste, u = heapq.heappop(abiertos)
        if u in cerrados:
            continue
        if u == META:
            return coste, expandidos, len(g)
        cerrados.add(u)
        expandidos += 1
        for v in sucesores(u):
            if coste + 1 < g.get(v, math.inf):
                g[v] = coste + 1
                heapq.heappush(abiertos, (coste + 1 + manhattan(v), coste + 1, v))
    return None, expandidos, len(g)


def paseo(pasos, semilla):
    rng, tablero = random.Random(semilla), META
    for _ in range(pasos):
        tablero = rng.choice(list(sucesores(tablero)))
    return tablero


tableros = [paseo(pasos, s) for pasos, s in [(20, 1), (40, 2), (80, 3), (200, 4), (400, 5), (1000, 6)]]
tableros.append(max(EXACTA, key=EXACTA.get))
print(f"{'óptimo':>7}{'expandidos A*':>15}{'en memoria':>12}")
resultados_a = []
for t in tableros:
    coste, expandidos, memoria = a_estrella(t)
    resultados_a.append((coste, expandidos, memoria))
    print(f"{coste:>7}{expandidos:>15}{memoria:>12}")
    assert coste == EXACTA[t]

181440 tableros alcanzables; distancia máxima: 31
 óptimo  expandidos A*  en memoria
     14             22          41
     18            283         465
     24           1593        2457
     24           2049        3141
     20            428         693
     26           2226        3409
     31          21197       30047


## 10. IDA\* frente a A\*

Escribir IDA\*: una búsqueda en profundidad que poda los nodos con $f = g + h$ mayor que una cota y, si no encuentra la meta, repite con la cota igual al menor $f$ podado. Guarda solo el camino actual (sin conjunto de visitados, salvo no volver al padre). Resolver los mismos tableros, comprobar que la longitud coincide con el BFS y comparar nodos expandidos y memoria con A\*.

In [10]:
def ida_estrella(inicio):
    expandidos = 0
    camino = [inicio]

    def buscar(g, cota):
        nonlocal expandidos
        u = camino[-1]
        f = g + manhattan(u)
        if f > cota:
            return f
        if u == META:
            return True
        expandidos += 1
        minimo = math.inf
        for v in sucesores(u):
            if len(camino) > 1 and v == camino[-2]:
                continue
            camino.append(v)
            r = buscar(g + 1, cota)
            if r is True:
                return True
            camino.pop()
            minimo = min(minimo, r)
        return minimo

    cota, iteraciones = manhattan(inicio), 0
    while True:
        iteraciones += 1
        r = buscar(0, cota)
        if r is True:
            return len(camino) - 1, expandidos, iteraciones, camino
        cota = r


print(f"{'óptimo':>7}{'A* expande':>12}{'IDA* expande':>14}{'iteraciones':>13}{'A* memoria':>12}{'IDA* memoria':>14}")
for t, (coste_a, exp_a, mem_a) in zip(tableros, resultados_a):
    coste, expandidos, iteraciones, camino = ida_estrella(t)
    print(f"{coste:>7}{exp_a:>12}{expandidos:>14}{iteraciones:>13}{mem_a:>12}{len(camino):>14}")
    assert coste == coste_a == EXACTA[t]
    assert all(b in set(sucesores(a)) for a, b in zip(camino, camino[1:])) and camino[-1] == META

 óptimo  A* expande  IDA* expande  iteraciones  A* memoria  IDA* memoria
     14          22            14            1          41            15
     18         283           234            6         465            19
     24        1593          2275            4        2457            25
     24        2049          2655            5        3141            25
     20         428           521            4         693            21
     26        2226          2089            5        3409            27
     31       21197         14195            6       30047            32


## 11. Binaria sobre la respuesta: maximizar el mínimo

En *Búsqueda binaria* (ejercicio 5) se minimizaba un máximo (la capacidad de un camión). Ahora al revés: en una playa hay torres en posiciones conocidas (en metros) y se deben colocar $k$ socorristas en torres distintas de forma que la **distancia mínima** entre dos de ellos sea lo mayor posible. El predicado «se pueden colocar con separación al menos $d$» es monótono y se comprueba de forma voraz. Comprobar contra todas las combinaciones de torres.

In [11]:
from itertools import combinations


def caben(torres, k, d):
    colocados, ultima = 1, torres[0]
    for t in torres[1:]:
        if t - ultima >= d:
            colocados, ultima = colocados + 1, t
    return colocados >= k


def maxima_separacion(torres, k):
    torres = sorted(torres)
    izq, der = 0, torres[-1] - torres[0]  # el último d que cabe
    while izq < der:
        medio = (izq + der + 1) // 2
        if caben(torres, k, medio):
            izq = medio
        else:
            der = medio - 1
    return izq


def fuerza_bruta(torres, k):
    return max(min(b - a for a, b in zip(c, c[1:])) for c in combinations(sorted(torres), k))


torres_las_canteras = [0, 180, 420, 610, 900, 1250, 1400, 1830, 2100, 2600, 3000]
for k in (2, 3, 4, 5, 6):
    d = maxima_separacion(torres_las_canteras, k)
    print(f"{k} socorristas: separación mínima de {d} m")
    assert d == fuerza_bruta(torres_las_canteras, k)

rng = random.Random(11)
for _ in range(300):
    torres = rng.sample(range(1000), rng.randint(2, 11))
    k = rng.randint(2, len(torres))
    assert maxima_separacion(torres, k) == fuerza_bruta(torres, k)

2 socorristas: separación mínima de 3000 m
3 socorristas: separación mínima de 1400 m
4 socorristas: separación mínima de 900 m
5 socorristas: separación mínima de 610 m
6 socorristas: separación mínima de 420 m


## 12. Binaria sobre una respuesta real

En la autopista hay estaciones de recarga en ciertos kilómetros y se pueden añadir $k$ más donde se quiera para que la **mayor separación** entre dos consecutivas sea mínima. Buscar la separación $d$ con bisección en reales (con un número fijo de iteraciones): con separación $d$, un tramo de longitud $L$ necesita $\lceil L/d \rceil - 1$ estaciones nuevas. Comprobar contra un método voraz exacto que añade cada estación al tramo cuyos trozos son ahora más largos.

In [12]:
def separacion_minima(estaciones, k, iteraciones=100):
    tramos = [b - a for a, b in zip(estaciones, estaciones[1:])]
    izq, der = 0.0, max(tramos)
    for _ in range(iteraciones):
        medio = (izq + der) / 2
        if sum(math.ceil(t / medio) - 1 for t in tramos) <= k:
            der = medio
        else:
            izq = medio
    return der


def separacion_voraz(estaciones, k):
    monticulo = [(-(b - a), b - a, 1) for a, b in zip(estaciones, estaciones[1:])]
    heapq.heapify(monticulo)
    for _ in range(k):
        _, tramo, partes = heapq.heappop(monticulo)
        heapq.heappush(monticulo, (-tramo / (partes + 1), tramo, partes + 1))
    return -monticulo[0][0]


estaciones = [0, 12, 31, 47, 80, 94, 130]
print(f"{'k':>3}{'bisección (km)':>16}{'voraz (km)':>12}")
for k in (0, 1, 3, 5, 10, 20):
    d1, d2 = separacion_minima(estaciones, k), separacion_voraz(estaciones, k)
    print(f"{k:>3}{d1:>16.4f}{d2:>12.4f}")
    assert math.isclose(d1, d2, rel_tol=1e-9)

rng = random.Random(12)
for _ in range(300):
    puntos = sorted(rng.sample(range(500), rng.randint(2, 12)))
    k = rng.randint(0, 30)
    assert math.isclose(separacion_minima(puntos, k), separacion_voraz(puntos, k), rel_tol=1e-9)

  k  bisección (km)  voraz (km)
  0         36.0000     36.0000
  1         33.0000     33.0000
  3         18.0000     18.0000
  5         16.0000     16.0000
 10          9.5000      9.5000
 20          5.5000      5.5000


## 13. El k-ésimo de una tabla de multiplicar

Dada la tabla de multiplicar de $m \times n$ (el valor de la celda $(i, j)$ es $i \cdot j$), encontrar el $k$-ésimo menor valor sin construirla: cuántos valores son $\le x$ se cuenta en $O(m)$ como $\sum_i \min(\lfloor x/i \rfloor, n)$, y ese recuento es monótono en $x$. Comprobar contra ordenar la tabla completa.

In [13]:
def k_esimo_tabla(m, n, k):
    izq, der = 1, m * n
    while izq < der:
        medio = (izq + der) // 2
        if sum(min(medio // i, n) for i in range(1, m + 1)) >= k:
            der = medio
        else:
            izq = medio + 1
    return izq


print("tabla 3×3, k=1..9:", [k_esimo_tabla(3, 3, k) for k in range(1, 10)])
print("tabla 30 000 × 40 000, k = 6·10⁸:", k_esimo_tabla(30_000, 40_000, 600_000_000))
rng = random.Random(13)
for _ in range(200):
    m, n = rng.randint(1, 30), rng.randint(1, 30)
    tabla = sorted(i * j for i in range(1, m + 1) for j in range(1, n + 1))
    k = rng.randint(1, m * n)
    assert k_esimo_tabla(m, n, k) == tabla[k - 1]

tabla 3×3, k=1..9: [1, 2, 2, 3, 3, 4, 6, 6, 9]


tabla 30 000 × 40 000, k = 6·10⁸: 224035686
